In [58]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import tushare as ts
import edhec_risk_kit as erk 
import time

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
hs300=erk.read_hs300_weights()
w=hs300.iloc[-2]
w=w.dropna()
w=w.sort_values(ascending=False)
h=w.index
w
end_date=w.name.strftime("%Y%m%d")
start_date=w.name-pd.DateOffset(years=10)
start_date=start_date.strftime("%Y%m%d")
    

In [ ]:

returns=erk.read_returns()
final_returns=returns.loc["2026-10"]
returns=returns.loc["2021-10":"2026-9"]
returns

,000001.SZ,000063.SZ,000100.SZ,000333.SZ,000338.SZ,000651.SZ,000725.SZ,000792.SZ,000858.SZ,002028.SZ,...,603259.SH,603501.SH,603986.SH,603993.SH,688008.SH,688012.SH,688041.SH,688072.SH,688256.SH,688981.SH
trade_date,,,,,,,,,,,,,,,,,,,,,
2021-10-08,0.039598,0.022034,0.035144,0.041667,0.002331,0.027355,0.051485,-0.027063,0.040248,-0.030081,...,-0.020812,-0.025267,0.032485,-0.004967,0.008416,0.015679,NaN,NaN,0.023235,0.010326
2021-10-11,0.040773,0.021855,-0.001543,-0.019586,0.002326,0.001005,-0.016949,-0.003434,-0.001840,-0.015898,...,-0.051865,-0.008500,-0.015698,0.056572,0.028209,-0.027111,NaN,NaN,0.006505,-0.006455
2021-10-12,-0.002577,-0.005780,0.013910,0.007597,-0.036543,-0.010289,0.001916,-0.012061,0.006760,-0.030720,...,0.008248,-0.034845,-0.035290,-0.003150,-0.038799,-0.038733,NaN,NaN,-0.027927,-0.016784
2021-10-13,0.011886,0.014244,-0.003049,-0.008657,0.010837,-0.008621,0.003824,0.013952,0.044257,-0.009290,...,0.041739,0.069465,0.021878,-0.011058,0.014693,0.010472,NaN,NaN,0.008029,-0.001652
2021-10-14,-0.018897,-0.026082,-0.019878,-0.012254,0.001787,-0.003069,-0.009524,0.010320,0.002130,0.031440,...,-0.034430,0.028345,0.005920,0.030351,-0.012816,-0.003363,NaN,NaN,-0.015184,-0.004045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-23,-0.009394,-0.019143,-0.019881,-0.000848,-0.042175,0.004715,0.003333,-0.015682,-0.002835,0.001246,...,0.000596,-0.001327,-0.008406,-0.008422,0.006942,-0.002028,-0.016953,0.010852,-0.015986,-0.008333
2026-09-24,-0.004405,-0.011152,-0.012170,0.002183,-0.022789,0.003650,-0.028239,-0.031863,-0.009525,-0.025980,...,-0.033046,-0.016552,-0.037495,-0.032843,-0.039710,-0.028768,-0.013351,-0.040257,-0.010516,-0.011615
2026-09-28,0.000000,-0.035401,-0.014374,-0.007864,-0.026877,-0.002597,-0.008547,-0.024051,-0.010909,-0.033142,...,-0.003695,-0.007985,-0.065821,-0.014637,-0.027506,-0.043619,-0.031414,-0.067199,-0.065489,-0.028505


In [74]:
miss=returns.isna().mean()
returns=returns.loc[:,miss<=0.05]
returns
cov=returns.cov()
codes=w.index[w.index.isin(returns.columns)][:50]
w=w[codes]
w=w/w.sum()
w
returns=returns[codes]
final_returns=final_returns[codes]
cov=returns.cov()
w_re=erk.equal_risk_contributions(cov)
w_re=pd.Series(w_re,index=cov.index)
w_re
n=len(final_returns)
cov=cov*n
n

17

In [87]:
rf=(1+0.02)**(n/252)-1
daily=(final_returns+1)@w
growth_vw=daily.prod()-1
cov_vw=w.T@cov@w
cov_vw=cov_vw**(0.5)

In [88]:
daily=(final_returns+1)@w_re

In [89]:

growth_re=daily.prod()-1
cov_re=w_re.T@cov@w_re
cov_re=cov_re**(0.5)

In [90]:
sp_vw=(growth_vw-rf)/cov_vw
sp_vw

np.float64(-2.104116826554997)

In [91]:
sp_re=(growth_re-rf)/cov_re
sp_re

np.float64(-0.9622058688417916)

In [92]:
check_w=erk.risk_contribution(w_re,cov)
check_w

300308.SZ    0.019984
600519.SH    0.020064
300750.SZ    0.020002
601318.SH    0.019917
300502.SZ    0.019973
600036.SH    0.020202
601899.SH    0.019992
603259.SH    0.020032
000333.SZ    0.020069
600900.SH    0.019965
688256.SH    0.019989
601166.SH    0.020085
601398.SH    0.019620
002371.SZ    0.019987
600030.SH    0.019977
002475.SZ    0.020070
688012.SH    0.019942
603986.SH    0.020006
688008.SH    0.019974
601138.SH    0.019974
688981.SH    0.019997
002594.SZ    0.019954
300059.SZ    0.019863
002384.SZ    0.020007
600276.SH    0.020042
000725.SZ    0.019980
601328.SH    0.020268
601288.SH    0.020136
601211.SH    0.020050
600183.SH    0.020002
600919.SH    0.019899
600150.SH    0.020072
600887.SH    0.020060
000651.SZ    0.020067
300394.SZ    0.020000
300408.SZ    0.020052
002463.SZ    0.020150
603993.SH    0.019870
601857.SH    0.020033
601816.SH    0.019880
000858.SZ    0.019974
000338.SZ    0.020036
601088.SH    0.019796
600309.SH    0.020091
300476.SZ    0.019952
600000.SH 